# 05c — Preprocessing sMRI: Volume Kanonik (v2)

**Tujuan.** Satu volume kanonik per subjek, `float32 (1, 96, 112, 96)` di ruang MNI152 2 mm, yang dipakai bersama oleh 2.5D CNN (irisan diambil saat persiapan input model, bukan di sini) dan 3D CNN. Tidak ada model yang dilatih, tidak ada preprocessing fenotipik, dan split tidak diubah.

**Metode (tidak berubah dari pilot pertama):** reorientasi RAS → N4 → HD-BET (GPU, venv terpisah, input LAS) → registrasi afin 12 DOF ke MNI152 1 mm → resample 2 mm → clip persentil 0,5–99,5 + z-score di dalam mask (latar 0) → crop box tetap + pad.

**Kode:** semua fungsi ada di `src/smri_preprocessing.py` dan dipakai bersama oleh notebook ini dan `src/run_05c.py` (terminal). Proses berat **disarankan lewat terminal** di luar VS Code:
```
python src/run_05c.py --mode pilot --dry-run
python src/run_05c.py --mode pilot --resume
```

**Input (read-only):** `data/folds.csv` (split v2), `data/pilot_ids.csv`, `data/exclusions.csv`, `data/audit_flags.csv`, master, `smri_qc.csv`.

**Output baru** di `data/smri_preproc/` (output lama di `data/vol`, `data/mask`, `data/vol_meta`, `data/work`, `data/failed_subjects.csv` **tidak disentuh**): `vol/`, `mask/`, `xfm/`, `meta/`, `qc_figures/`, `templates/`, `logs/`, `runs/`, `qc_report.csv`, `failed_subjects.csv`, `status.csv`, `qc_review.csv`, `qc_decisions_template.csv`, `nii/`, `mask_native/`, `header_check.csv`, `preprocess_config.json`, `reference_space.json`, `hdbet_runner_validation.json`.

**File kerja:** `C:\ta_work\05c` (di luar OneDrive): checkpoint per langkah per subjek, chunk HD-BET, validasi, cache, `run.lock`, `STOP`.

| CELL | Isi | Berat? |
|---|---|---|
| 1 | CONFIG + flag | – |
| 2 | Lingkungan dan preflight resource | ringan |
| 3 | Daftar subjek + audit warisan (`failed_subjects.csv` lama vs pilot baru) | ringan |
| 4 | Template, ruang referensi (shape, spacing, affine) | ringan |
| 5 | Cek header citra → `header_check.csv` | baca 1.002 header (±3 menit pertama kali) |
| 6 | Sidik jari konfigurasi + status checkpoint | ringan |
| 7 | Validasi runner vs CLI (`RUN_VALIDATION`) | HD-BET 1–2× |
| 8 | Pilot (`RUN_PILOT_IN_NOTEBOOK`, disarankan via terminal) | berat |
| 9 | QC output + galeri | ringan |
| 10 | QC blind (tanpa label) | – |
| 11 | Verifikasi akhir (hash input) | ringan |

### CELL 1 — CONFIG

**Tujuan:** konfigurasi tunggal (`sp.Config`), seed 42, flag proses berat. Parameter metode dan ambang ada di `src/smri_preprocessing.py::Config` dan dicetak di sini. **Output:** `cfg`.

In [ ]:
import importlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "dataset").exists() and (PROJECT_ROOT.parent / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
import smri_preprocessing as sp
sp = importlib.reload(sp)

cfg = sp.Config(project_root=PROJECT_ROOT)       # WORK_DIR default C:\ta_work\05c (env TA05C_WORK_DIR)
SEED = cfg.seed
np.random.seed(SEED)

RUN_VALIDATION = False            # Tahap 2: validasi runner vs CLI pada VALIDATION_SUB_ID
RUN_PILOT_IN_NOTEBOOK = False     # Tahap 3: lebih aman lewat terminal (src/run_05c.py)
RUN_QC_ANALYSIS = False           # setelah batch penuh
VALIDATION_SUB_ID = 50002

assert SEED == 42 and cfg.reg_transform == "Affine" and tuple(cfg.target_shape) == (96, 112, 96)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("WORK_DIR    :", cfg.work_dir)
print("OUTPUT      :", cfg.out_dir)
print("Metode      :", json.dumps(cfg.method_params(), default=str))
print(f"Eksekusi    : HD-BET {cfg.hdbet_mode} (proses {cfg.hdbet_np_preprocessing}+{cfg.hdbet_np_export}), chunk {cfg.chunk_size}, "
      f"max retry {cfg.max_retries}, RAM guard {cfg.min_free_ram_gb} GiB, RAM mulai ≥ {cfg.min_start_ram_gb} GiB")

### CELL 2 — Lingkungan dan preflight resource

**Tujuan:** mendeteksi versi tool (kernel dan venv HD-BET), GPU, hash bobot HD-BET, lalu preflight RAM/VRAM/disk untuk pilot. Tidak ada yang ditulis di cell ini. **Output:** `ENV`, `PREFLIGHT`.

In [ ]:
ENV = sp.env_versions(cfg, write_cache=False)
venv = ENV["hdbet_venv"]
rows = [(k, ENV.get(k)) for k in ("python", "antspyx", "nibabel", "numpy", "nilearn", "psutil")]
rows += [(f"venv {k}", venv.get(k)) for k in ("hd-bet", "nnunetv2", "torch", "torch_cuda_build", "cuda_available", "gpu")]
rows += [("bobot HD-BET", (ENV.get("hdbet_weights") or {}).get("sha256", "TIDAK ADA")[:16])]
display(pd.DataFrame(rows, columns=["komponen", "versi/status"]))
if not venv.get("hd-bet"):
    raise RuntimeError("HD-BET tidak ada di .venv-hdbet. Lihat instruksi instalasi di notebook versi sebelumnya.")
if cfg.hdbet_device == "cuda" and not venv.get("cuda_available"):
    raise RuntimeError("torch di venv HD-BET tanpa CUDA. Tidak ada fallback otomatis ke CPU.")
PREFLIGHT_OK, PREFLIGHT = sp.preflight(cfg, n_subjects=20)
print(json.dumps(PREFLIGHT, indent=1, default=str))

### CELL 2b — Verifikasi CELL 2

In [ ]:
assert ENV["antspyx"] and venv.get("hd-bet") and venv.get("cuda_available")
print("Preflight pilot:", "OK" if PREFLIGHT_OK else "TIDAK LOLOS → " + "; ".join(PREFLIGHT["problems"]))

### CELL 3 — Daftar subjek dan audit warisan

**Tujuan:** membaca `folds.csv` (split v2) dan `pilot_ids.csv` terbaru (05c tidak membuat split), memastikan subjek eksklusi tidak ikut, dan mengaudit ketidaksesuaian `data/failed_subjects.csv` lama terhadap pilot baru (read-only). **Output:** `SUBJ_ALL`, `SUBJ_PILOT`, `LEGACY`.

In [ ]:
SUBJ_ALL = sp.load_subjects(cfg, "all")
SUBJ_PILOT = sp.load_subjects(cfg, "pilot")
EXCL = set(pd.read_csv(cfg.paths["exclusions"])["SUB_ID"])
print(f"Semua: {len(SUBJ_ALL)} (dev {int((SUBJ_ALL['split'] == 'dev').sum())}, test {int((SUBJ_ALL['split'] == 'test').sum())}) | "
      f"eksklusi: {sorted(EXCL)}")
print(f"Pilot: {len(SUBJ_PILOT)} | situs {SUBJ_PILOT['SITE_ID'].nunique()} | "
      f"QC {SUBJ_PILOT['qc_decision'].value_counts().to_dict()}")
print("subset200:", "ada" if cfg.paths["subset200"].exists() else "BELUM ADA (dibuat di 05a, perlu persetujuan)")
LEGACY = sp.audit_legacy(cfg, SUBJ_PILOT)
print(json.dumps(LEGACY, indent=1))

### CELL 3b — Verifikasi CELL 3

In [ ]:
assert not EXCL & set(SUBJ_ALL["SUB_ID"]) and len(SUBJ_ALL) == 1002
assert len(SUBJ_PILOT) == 20 and (SUBJ_PILOT["split"] == "dev").all()
assert SUBJ_ALL["smri_abspath"].map(Path.exists).all(), "Ada file citra yang hilang"
print("Daftar subjek OK | file citra semua ada | pilot ⊆ dev | eksklusi tidak ikut")

### CELL 4 — Template dan ruang referensi

**Tujuan:** template MNI152 nilearn (ditulis ke `data/smri_preproc/templates/` hanya jika belum ada; tidak menimpa), box crop tetap, dan affine grid akhir. **Output:** `SPACE`, `REF` (juga `reference_space.json` bila `RUN_*` aktif).

In [ ]:
SPACE = sp.prepare_space(cfg, write=True)
REF = sp.reference_space_info(cfg, SPACE)
print(f"Shape akhir : {REF['shape']} ({REF['dtype']}) | axcodes {REF['axcodes']} | spacing {REF['spacing_mm']} mm")
print("Affine akhir:\n", np.array(REF["affine"]))
print("Crop:", REF["crop"], "| pad:", REF["pad"])

### CELL 4b — Verifikasi CELL 4

In [ ]:
assert REF["axcodes"] == "RAS" and np.allclose(REF["spacing_mm"], [cfg.target_res_mm] * 3)
assert SPACE.template_mask_crop.shape == tuple(cfg.target_shape)
print("Mask template utuh setelah crop:", int(SPACE.template_mask_crop.sum()), "voxel")

### CELL 5 — Cek header citra → `header_check.csv`

**Tujuan:** dimensi, voxel size, orientasi, dtype, NaN/Inf, intensitas negatif untuk semua subjek non-eksklusi. Baris `data/smri_header_check.csv` lama dipakai sebagai cache **read-only** bila file citra tidak berubah. **Output:** `HDR`, `data/smri_preproc/header_check.csv`.

In [ ]:
HDR = sp.header_check(cfg, SUBJ_ALL, legacy_cache=cfg.data_dir / "smri_header_check.csv")
cfg.out_dir.mkdir(parents=True, exist_ok=True)
sp.write_csv_atomic(cfg.paths["header_check"], HDR)
print("Orientasi asli:", HDR["axcodes"].value_counts().to_dict())
print("Anisotropi > 1,5:", int((HDR["anisotropy"] > 1.5).sum()), "| voksel negatif (subjek):", int((HDR["n_negative"].fillna(0) > 0).sum()))
print("Anomali yang menghalangi:", int(HDR["flag_blocking"].sum()))

### CELL 5b — Verifikasi CELL 5

In [ ]:
assert len(HDR) == len(SUBJ_ALL) and HDR["SUB_ID"].is_unique
assert not HDR["flag_blocking"].any(), HDR.loc[HDR["flag_blocking"], ["SUB_ID", "error"]]

### CELL 6 — Sidik jari konfigurasi dan status checkpoint

**Tujuan:** menghitung sidik jari (versi pipeline, parameter metode, crop, versi antspyx/nibabel/numpy/nilearn/HD-BET/nnunetv2/torch/SimpleITK, hash template, hash bobot HD-BET). Output hanya dilewati jika valid dan sidik jarinya sama. **Output:** `FP`, `STATE`.

In [ ]:
FP, FP_PAYLOAD = sp.config_fingerprint(cfg, SPACE, ENV)
FAILED = sp.FailedRegistry(cfg)
STATE = pd.DataFrame([{"SUB_ID": int(s), "valid": sp.output_valid(cfg, s, FP),
                       "checkpoint": [k for k in ("reorient", "n4", "skullstrip", "register")
                                      if (sp.subject_wd(cfg, s) / f"{k}.done.json").exists()],
                       "failed_attempts": FAILED.attempts(s, FP)} for s in SUBJ_PILOT["SUB_ID"]])
print("Sidik jari konfigurasi:", FP)
display(STATE)

### CELL 7 — Validasi runner vs CLI (Tahap 2)

**Tujuan:** pada `VALIDATION_SUB_ID`, bandingkan mask HD-BET runner vs CLI dan volume akhir dari keduanya. Lolos jika shape/affine sama, Dice ≥ 0,999 (atau bitwise identik), max|Δ| volume ≤ 1e-3, korelasi ≥ 0,9999. Mask CLI lama di `data/work/hdbet_validation` hanya **disalin** (hash diverifikasi) bila input HD-BET-nya identik; jika tidak, CLI dijalankan sekali. **Output:** `data/smri_preproc/hdbet_runner_validation.json`.

In [ ]:
if RUN_VALIDATION:
    ok, pre = sp.preflight(cfg, 1)
    if not ok:
        raise RuntimeError("Preflight gagal: " + "; ".join(pre["problems"]))
    cfg.ensure_dirs()
    with sp.RunLock(cfg):
        VALIDATION = sp.validate_runner_vs_cli(cfg, FP, SPACE, VALIDATION_SUB_ID)
    print(json.dumps(VALIDATION, indent=1, default=str))
    if not VALIDATION["passed"]:
        raise RuntimeError("Validasi runner GAGAL; pilot tidak boleh dijalankan.")
else:
    p = cfg.out_dir / "hdbet_runner_validation.json"
    VALIDATION = json.loads(p.read_text(encoding="utf-8")) if p.exists() else None
    print("RUN_VALIDATION = False |", "hasil tersimpan: " + str(VALIDATION.get("passed")) if VALIDATION else "belum ada hasil validasi")

### CELL 8 — Pilot (Tahap 3)

**Tujuan:** memproses `pilot_ids.csv` dengan resume. Disarankan dari terminal: `python src/run_05c.py --mode pilot --resume`. Cell ini hanya menjalankannya bila `RUN_PILOT_IN_NOTEBOOK = True` **dan** validasi sudah lolos. **Output:** output pilot di `data/smri_preproc/`.

In [ ]:
if RUN_PILOT_IN_NOTEBOOK:
    assert VALIDATION and VALIDATION.get("passed"), "Validasi runner vs CLI belum lolos"
    import run_05c
    code = run_05c.main(["--mode", "pilot", "--resume"])
    print("Kode keluar run_05c:", code)
else:
    print("RUN_PILOT_IN_NOTEBOOK = False. Jalankan dari terminal: python src/run_05c.py --mode pilot --resume")

### CELL 9 — QC output + galeri

**Tujuan:** untuk semua subjek dengan output valid (sidik jari sama): shape, NaN/Inf, fraksi nol, NCC dengan template, rasio dan Dice mask terhadap template, volume otak native, determinan registrasi, z-score outlier (global n ≥ 20, per situs n ≥ 15), dan flag (hanya penanda, tidak membuang). Galeri: 12 acak + semua yang ber-flag. **Output:** `qc_report.csv`, galeri PNG.

In [ ]:
QC = sp.build_qc_report(cfg, SPACE, SUBJ_ALL, FP)
if len(QC):
    sp.write_csv_atomic(cfg.paths["qc_report"], QC)
    cols = ["SUB_ID", "SITE_ID", "status", "pcp_qc_decision", "ncc_template", "mask_ratio_template", "dice_mask_template",
            "volume_mask_ml", "reg_det", "n_flags", "flag_reasons"]
    display(QC[cols].round(3))
    rng = np.random.default_rng(SEED)
    rand_ids = sorted(rng.choice(QC["SUB_ID"], size=min(12, len(QC)), replace=False).tolist())
    sp.gallery(cfg, SPACE, rand_ids, QC, cfg.paths["fig"] / "gallery_random.png", "12 subjek acak")
    flagged = QC.loc[QC["n_flags"] > 0, "SUB_ID"].tolist()
    for k in range(0, len(flagged), 24):
        sp.gallery(cfg, SPACE, flagged[k:k + 24], QC, cfg.paths["fig"] / f"gallery_flagged_{k // 24:02d}.png",
                   f"Subjek ber-flag QC ({len(flagged)})")
    display(Image(filename=str(cfg.paths["fig"] / "gallery_random.png")))
else:
    print("Belum ada output valid untuk sidik jari", FP)

### CELL 9b — Verifikasi CELL 9

In [ ]:
if len(QC):
    assert QC["SUB_ID"].is_unique and (QC["n_nonfinite"] == 0).all()
    assert (QC["final_shape"] == str((1, *cfg.target_shape))).all()
    assert (QC["config_fp"] == FP).all()
    print(f"QC: {len(QC)} subjek | ber-flag {int((QC['n_flags'] > 0).sum())} |",
          QC[[c for c in QC.columns if c.startswith("flag_")]].sum().to_dict())

### CELL 10 — QC blind

**Tujuan:** menegaskan bahwa QC dilakukan tanpa label. Review memakai `qc_review.csv` (tanpa diagnosis); keputusan di `qc_decisions.csv`. **Output:** tidak ada.

In [ ]:
### CELL 10 — (dihapus) analisis QC vs label
# QC blind: label/diagnosis tidak boleh dipakai di tahap QC. Keputusan QC dicatat di
# data/smri_preproc/qc_decisions.csv SEBELUM label digabung untuk training.
print('QC blind: tidak ada analisis QC vs label di 05c.')


### CELL 11 — Verifikasi akhir

**Tujuan:** memastikan input read-only (master, `folds.csv`, `pilot_ids.csv`, `exclusions.csv`) tidak berubah oleh notebook ini, dan output lama tidak tersentuh.

In [ ]:
H_EXPECTED = json.loads((cfg.out_dir / "input_hashes.json").read_text()) if (cfg.out_dir / "input_hashes.json").exists() else None
H_NOW = {k: sp.sha256_file(cfg.paths[k]) for k in ("master", "folds", "pilot", "exclusions")}
if H_EXPECTED is None:
    sp.write_json_atomic(cfg.out_dir / "input_hashes.json", H_NOW)
    print("Hash input dicatat pertama kali:", {k: v[:12] for k, v in H_NOW.items()})
else:
    changed = [k for k in H_NOW if H_NOW[k] != H_EXPECTED.get(k)]
    print("Hash input:", "tidak berubah" if not changed else f"BERUBAH: {changed}")
    assert not changed, "Input split/master berubah sejak run sebelumnya; periksa 05a sebelum melanjutkan."
legacy_vol = sorted(p.name for p in (cfg.data_dir / "vol").glob("*.npy"))
print("Output lama data/vol (tidak disentuh):", legacy_vol)